# Residual stream: attn write → add → MLP write → add

**Previous:** [02_hidden_states](02_hidden_states.ipynb)  
**Home:** [../00_START_HERE.ipynb](../00_START_HERE.ipynb)  
**Next:** [04_attention](04_attention.ipynb)

**Kernel:** `CXR local Qwen (faiss_gpu1)`  
**Status:** executable (calls `scripts/` — same as CLI)

---

## 1. Why am I learning this?

A block does not replace the residual — it **writes** attn and MLP vectors and **adds** them.

## 2. Mental model

residual_in → attn write → ADD → MLP write → ADD → residual_out (same width).

## 3. Clinical question

On layer 0, what are the L2 sizes of residual / attn / MLP writes?

## 4. Prediction

Attn and MLP writes have non-zero L2; add-check near 0.


## 5. Minimal Python — setup + load (run once)


In [22]:
# Shared bootstrap — run this first in every executable notebook
import sys
from pathlib import Path

# notebooks/_lib regardless of how deep this notebook sits
_here = Path.cwd().resolve()
for _p in [_here, *_here.parents]:
    if (_p / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "_lib"))
        break
    if (_p / "notebooks" / "_lib" / "cxr_boot.py").is_file():
        sys.path.insert(0, str(_p / "notebooks" / "_lib"))
        break
else:
    raise RuntimeError("Cannot find notebooks/_lib/cxr_boot.py — open Jupyter with notebooks/ as root")

import cxr_boot
ctx = cxr_boot.setup(layer=20, max_new=24, load_model=True)
model, tok = ctx["model"], ctx["tok"]
NOTE, LAYER, MAX_NEW = ctx["NOTE"], ctx["LAYER"], ctx["MAX_NEW"]
PROMPT_A, PROMPT_B, PROMPT_TEST = ctx["PROMPT_A"], ctx["PROMPT_B"], ctx["PROMPT_TEST"]
look, intervene, process = ctx["look"], ctx["intervene"], ctx["process"]
print("NOTE:", NOTE)
print("LAYER:", LAYER, "ready")


BACKEND  REUSE — not calling from_pretrained; inspecting live weights
  pid=4097531  class=Qwen2ForCausalLM
  name=Qwen/Qwen2.5-7B-Instruct
  torch_dtype=torch.float16  training=False
  blocks=28  hidden=3584  using layer 20
  tokenizer=Qwen2TokenizerFast  vocab=151643  pad=151643  eos=151645
  hf_device_map (accelerate placement):
    model.embed_tokens: 0
    model.layers.0: 0
    model.layers.1: 0
    model.layers.2: 0
    model.layers.3: 0
    model.layers.4: 0
    model.layers.5: 0
    model.layers.6: 0
    model.layers.7: 0
    model.layers.8: 0
    model.layers.9: 0
    model.layers.10: 0
    model.layers.11: 0
    model.layers.12: 0
    model.layers.13: 0
    model.layers.14: 0
    model.layers.15: 0
    model.layers.16: 0
    model.layers.17: 0
    model.layers.18: 0
    model.layers.19: 0
    model.layers.20: 0
    model.layers.21: 0
    model.layers.22: 0
    model.layers.23: 0
    model.layers.24: 0
    model.layers.25: 0
    model.layers.26: 0
    model.layers.27: cpu
    

#### <u>Explanation: 5. Minimal Python – setup + load (run once)</u>

Yes. This is actually a **very good bootstrap result**. The important thing to understand is that this cell has **not yet performed a mechanistic-interpretability experiment**. It has prepared your laboratory.

Think of the output as saying:
> **“I found the already-loaded Qwen model, inspected its architecture and memory placement, selected Layer 20, loaded your test note and tools, and I'm ready for experiments.”**

### What the code is doing

The first half of the cell is mostly plumbing. It searches upward through your directories until it finds:

```python
_lib/cxr_boot.py
```

and adds that `_lib` directory to Python's import path. That allows:

```python
import cxr_boot
```
to work regardless of which notebook subfolder you're currently in. Then the important line is:

```python
ctx = cxr_boot.setup(layer=20, max_new=24, load_model=True)
```

That creates your experimental environment. You subsequently unpack it:

```python
model, tok = ctx["model"], ctx["tok"]
```

So `model` is your live Qwen transformer and `tok` is its tokenizer.

You also retrieve:

```python
NOTE
LAYER
MAX_NEW
PROMPT_A
PROMPT_B
PROMPT_TEST
look
intervene
process
```
Those last three are particularly important conceptually:

```text
look       → observe the model
intervene  → manipulate the model
process    → supporting processing
```

That's a nice architecture for an MI notebook because it naturally separates **observation from causal intervention**.
---

## Now let's decode your output

The most important line is:

```text
BACKEND REUSE — not calling from_pretrained;
inspecting live weights
```

This is excellent.

Normally Hugging Face might do:

```python
AutoModelForCausalLM.from_pretrained(...)
```

which would load another copy of your ~7.6-billion-parameter model.

Your bootstrap instead discovered the **model already sitting in memory** and reused it.

That's why your notebooks can start quickly without reloading 14+ GB of weights every time.

Then:

```text
pid=4097531
```

That's simply the operating-system process ID containing the model.

---

### Your actual model

```text
class=Qwen2ForCausalLM
name=Qwen/Qwen2.5-7B-Instruct
```

You're working with **Qwen2.5-7B-Instruct**.

Then:

```text
torch_dtype=torch.float16
```

means most model weights are represented using FP16: 16-bit floating-point numbers.

And:

```text
training=False
```

means the model is in inference/evaluation mode rather than being trained.

That's what you want for these experiments.

---

# This part is especially important for learning MI

```text
blocks=28
hidden=3584
using layer 20
```

This tells you about the actual transformer you're examining.

You have:

```text
                    Qwen2.5-7B

Input tokens
     │
     ▼
Embeddings
     │
     ▼
Layer 0
     │
Layer 1
     │
Layer 2
     │
 ...
     │
Layer 19
     │
     ▼
┌─────────────────┐
│    LAYER 20     │  ← your current inspection site
│                 │
│   Attention     │
│       ↓         │
│      MLP        │
└─────────────────┘
     │
Layer 21
     │
 ...
     │
Layer 27
     │
     ▼
Final Norm
     │
     ▼
LM Head
     │
     ▼
151,643-ish
token scores
```

There are **28 transformer blocks**, numbered:

```text
0 → 27
```

and you've chosen:

```text
Layer 20
```

for your current experiments.

---

# And here's an important correction to something we discussed earlier

Your current model's residual stream is **not 4096 dimensions**.

Your bootstrap has actually inspected the model and reports:

```text
hidden=3584
```

So for **this Qwen2.5-7B model**, each token's residual representation is a vector containing **3,584 values**.

For example:

```text
"Patient"

Layer 0
[0.17, -0.81, 0.03, .............]
               3,584 numbers

              ↓

Layer 1
[0.31, -0.72, 0.19, .............]
               3,584 numbers

              ↓
             ...

              ↓

Layer 20
[-0.12, 0.93, 0.47, .............]
               3,584 numbers
```

That's exactly the kind of fact your notebook should display prominently, because it grounds the abstract concepts in **your actual model** rather than generic transformer examples.

---

# Your tokenizer

You have:

```text
tokenizer=Qwen2TokenizerFast
vocab=151643
```

The tokenizer converts your text into token IDs.

Conceptually:

```text
Patient received FOLFOX.
        ↓
     tokenizer
        ↓
[token1, token2, token3, token4, ...]
        ↓
     embeddings
        ↓
3584-dimensional vectors
        ↓
     transformer
```

The vocabulary figure means the tokenizer has roughly 151,643 ordinary token IDs available in its configured vocabulary.

You also see:

```text
pad=151643
eos=151645
```

These are special token IDs used for padding and end-of-sequence behavior. Special-token IDs can sit outside the ordinary `vocab_size` count depending on tokenizer configuration, so those numbers aren't necessarily contradictory.

---

# Now the GPU section

This looks intimidating:

```text
hf_device_map (accelerate placement):
```

but it's actually straightforward.

Your GPU is numbered:

```text
0
```

So:

```text
model.layers.0: 0
model.layers.1: 0
...
model.layers.26: 0
```

means:

> Layers 0–26 are assigned to GPU 0.

But:

```text
model.layers.27: cpu
model.norm: cpu
model.rotary_emb: cpu
lm_head: cpu
```

means some of the final components have been offloaded to system RAM/CPU.

Visually:

```text
RTX 3090
────────────────────────

Embeddings       GPU
L0               GPU
L1               GPU
L2               GPU
...
L20              GPU   ← YOUR EXPERIMENT
...
L26              GPU

────────────────────────
CPU

L27              CPU
Final Norm       CPU
Rotary component CPU
LM Head          CPU
```

That's why you're able to fit the model despite its size.

And importantly, **Layer 20 is on the GPU**, which is convenient for your interventions.

---

# Then you have the model size

```text
params=7.616B
```

Approximately:

**7.616 billion parameters.**

And:

```text
weight_GiB=14.19
```

makes sense because FP16 uses roughly two bytes per parameter:

$$
7.616B \times 2 \approx 15.23\text{ GB}
$$

which is about **14.2 GiB** after converting decimal GB to binary GiB.

So your notebook is actually teaching you something useful already:

```text
7.616 billion parameters
        ×
2 bytes/parameter
        ≈
15.2 GB
        ≈
14.2 GiB
```

---

# Your GPU memory

You have:

```text
cuda_alloc_GiB=12.74
cuda_reserved_GiB=13.79
cuda0 free_GiB=8.56
total_GiB=23.56
```

Your 3090 has roughly **24 GB physical VRAM** (reported here as 23.56 GiB).

PyTorch currently has about:

```text
12.74 GiB actively allocated
```

and has reserved approximately:

```text
13.79 GiB
```

for its CUDA memory allocator.

You still have:

```text
8.56 GiB free
```

That's useful because MI experiments can create substantial temporary activation tensors.

---

# One slightly strange-looking thing

You have:

```text
param tensors by device:
{'cuda:0': 325, 'meta': 14}
```

You might reasonably ask:

> Why does it say `meta` if some components are supposedly on CPU?

A `meta` tensor is not an ordinary CPU tensor. It's effectively a lightweight placeholder that describes a tensor's shape/type without necessarily storing its full data there. Hugging Face Accelerate can use these placeholders as part of model dispatch/offloading.

So don't read:

```text
meta = missing model
```

from this output.

Your bootstrap is explicitly reporting Accelerate's dispatch setup, and the model is operational.

---

# Finally, your actual experimental input

Your notebook says:

```text
NOTE:
Patient received FOLFOX.
Disease progressed.
FOLFOX was discontinued.
```

That is now the text you're going to put through the transformer.

And:

```text
LAYER: 20 ready
```

means:

> **Layer 20 has been selected as the default site for subsequent LOOK/intervention experiments.**

It does **not** mean Layer 20 has been analyzed yet.

That's the key distinction.

Right now you're here:

```text
                    SETUP
                      │
                      ▼
             Model loaded/reused ✓
                      │
             Tokenizer available ✓
                      │
              Note available ✓
                      │
             Layer 20 selected ✓
                      │
                      ▼
              ┌──────────────┐
              │ WE ARE HERE  │
              └──────┬───────┘
                     │
          ┌──────────┴──────────┐
          ▼                     ▼
        LOOK                 INTERVENE
          │                     │
          ▼                     ▼
   What is happening?     Change something
          │                     │
          └──────────┬──────────┘
                     ▼
                 MEASURE
                     │
                     ▼
                  EXPLAIN
```

And **this is exactly where I would start improving the notebook**.

Instead of leaving that huge bootstrap dump as raw text, your next notebook section should turn these facts into a clean **"Model Dashboard"** showing something like:

```text
╔══════════════════════════════════════════════╗
║              MODEL UNDER STUDY              ║
╠══════════════════════════════════════════════╣
║ Qwen2.5-7B-Instruct                         ║
║ Parameters          7.616 B                 ║
║ Transformer blocks  28                      ║
║ Residual width      3,584                   ║
║ Vocabulary          151,643                 ║
║ Precision           FP16                    ║
║ Target layer        L20                     ║
║ GPU                 RTX 3090 ~24 GB         ║
╚══════════════════════════════════════════════╝

               CURRENT NOTE

 Patient received FOLFOX.
 Disease progressed.
 FOLFOX was discontinued.

                     ↓

 Tokenize → Embed → L0 → ... → L20 → ... → L27 → Logits
                              ▲
                              │
                       microscope here
```

Then the **very next educational notebook cell** I'd build is not SAE, steering, patching, or circuits.

It should be:

> **"What actually happens to this exact sentence as it passes through Qwen?"**

Tokenize this `NOTE`, show every token, its position, and then capture the residual stream at **every layer for every token**. That becomes the foundation on which essentially everything else in your MI/RepEng notebook can be built.

_(fill after you run)_

## 9. Claim boundary

✓ We measured residual-plus-writes on this note.

✗ L2 size is not importance; we have not ablated anything.

## 10. CXR connection

`look.py --mode sites` · learning lab E00.3

## 11. Questions

## 12. Revision notes

| Date | Change |
|------|--------|
| | |


## 6. Run


In [4]:
# Same sites as look.py --mode sites, plus explicit add path on layer 0
look.cmd_sites(model, tok, layer=0, text=NOTE)

from _common import blocks, unwrap, encode, device
import torch

ids = encode(tok, NOTE, device(model))
block = blocks(model)[0]
bucket = {}

def pre(_m, inp):
    hs = unwrap(inp[0] if isinstance(inp, tuple) else inp)
    bucket["rin"] = hs[0, -1, :].detach().float().cpu()

def attn_h(_m, _i, out):
    bucket["attn"] = unwrap(out)[0, -1, :].detach().float().cpu()

def mlp_h(_m, _i, out):
    bucket["mlp"] = unwrap(out)[0, -1, :].detach().float().cpu()

def post(_m, _i, out):
    bucket["rout"] = unwrap(out)[0, -1, :].detach().float().cpu()

handles = [
    block.register_forward_pre_hook(pre),
    block.self_attn.register_forward_hook(attn_h),
    block.mlp.register_forward_hook(mlp_h),
    block.register_forward_hook(post),
]
try:
    with torch.inference_mode():
        model(**ids)
finally:
    for h in handles:
        h.remove()

rin, attn, mlp, rout = bucket["rin"], bucket["attn"], bucket["mlp"], bucket["rout"]
print("resid_in L2", float(rin.norm()))
print("attn write L2", float(attn.norm()))
print("after attn L2", float((rin + attn).norm()))
print("MLP write L2", float(mlp.norm()))
print("resid_out L2", float(rout.norm()))
print("add_check ||rin+attn+mlp-rout||", float((rin + attn + mlp - rout).norm()))
print("Path: residual → attention write → add → MLP write → add → next residual")


L0 block  dim=3584  L2=10.4372
L0 attn   dim=3584  L2=7.3870
L0 mlp    dim=3584  L2=4.5942
resid_in L2 0.8561864495277405
attn write L2 7.387022495269775
after attn L2 7.499487400054932
MLP write L2 4.594157695770264
resid_out L2 10.437214851379395
add_check ||rin+attn+mlp-rout|| 0.0032837833277881145
Path: residual → attention write → add → MLP write → add → next residual


### Explanation for 6 Run 

Yes. And there is one **very important source of confusion** here:

**`L2` in your printed output does NOT mean Layer 2.**

It means the **L2 norm** — the length/magnitude of a vector. You are actually examining **Layer 0**.

Your code says:

```python
block = blocks(model)[0]
```

So everything below is from **transformer block Layer 0**.

### What you captured

For the **last token of your NOTE**, Layer 0 is essentially doing this:

```text
resid_in
3584-dimensional vector
|| · ||₂ = 0.856
       │
       ▼
   ATTENTION
       │
       │ writes another 3584-D vector
       │ ||attn||₂ = 7.387
       ▼
resid_in + attn
|| · ||₂ = 7.499
       │
       ▼
      MLP
       │
       │ writes another 3584-D vector
       │ ||mlp||₂ = 4.594
       ▼
resid_out
|| · ||₂ = 10.437
```

That's the fundamental residual-stream operation you're trying to understand.

### Now your numbers

Your first three lines come from:

```python
look.cmd_sites(...)
```

which reports:

```text
L0 block  dim=3584  L2=10.4372
L0 attn   dim=3584  L2=7.3870
L0 mlp    dim=3584  L2=4.5942
```

Here `L0` = **Layer 0**.

But `L2=10.4372` = **L2 norm**, not Layer 2.

So read the first line as:

> At Layer 0, the block's output vector has 3,584 dimensions and has an L2 magnitude of 10.4372.

Your second piece of code then opens up that same block to show how it got there.

---

### 1. `resid_in L2 = 0.856`

```python
bucket["rin"]
```

is the residual vector **entering Layer 0**, at the last token.

It's:

```text
3584 numbers

[x₁, x₂, x₃, ... x₃₅₈₄]
```

Instead of printing all 3,584 numbers, you're calculating:

```python
rin.norm()
```

which reduces the entire vector to one number describing its magnitude:

**0.856**

Think of it as the length of an arrow in 3,584-dimensional space.

---

### 2. Attention writes `7.387`

Attention processes that residual representation and produces another 3,584-D vector:

```text
ATTENTION WRITE

[a₁, a₂, a₃, ... a₃₅₈₄]

magnitude = 7.387
```

This is the **attention contribution** that gets added to the residual stream.

So:

$$r_{\text{after-attn}} = r_{\text{in}} + a$$

Your code explicitly calculates that:

```python
rin + attn
```

and obtains:

```text
after attn L2 = 7.499
```

So:

```text
          residual entering L0
                 0.856
                   │
                   │
                   +  attention write
                      7.387
                   │
                   ▼
          residual after attention
                 7.499
```

Notice something important:

**0.856 + 7.387 ≠ 7.499**

That's not an error.

These are **vector magnitudes**, not ordinary scalar quantities.

You are actually adding:

```text
3584-dimensional vector
        +
3584-dimensional vector
```

and then measuring the magnitude of the resulting vector.

---

### 3. MLP writes `4.594`

Then the MLP processes the attention-updated residual and produces its own 3,584-D contribution:

```text
MLP WRITE

[m₁, m₂, m₃, ... m₃₅₈₄]

magnitude = 4.594
```

That contribution is added too.

Conceptually:

$$r_{\text{out}} = r_{\text{in}} + \text{attention} + \text{MLP}$$

giving:

```text
resid_out L2 = 10.437
```

So your complete Layer 0 is approximately:

```text
             LAYER 0

Residual IN
||r|| = 0.856
      │
      │
      ├────► ATTENTION
      │        │
      │        │ write = 7.387
      │◄───────┘
      │
      ▼
Residual after attention
||r|| = 7.499
      │
      │
      ├────► MLP
      │        │
      │        │ write = 4.594
      │◄───────┘
      │
      ▼
Residual OUT
||r|| = 10.437
      │
      ▼
     L1
```

That's actually a very nice little experiment because you're **watching the residual stream being constructed**.

### And your `add_check` is particularly useful

You calculated:

```python
rin + attn + mlp - rout
```

and got:

```text
0.00328
```

You're asking:

> If I manually add the incoming residual + attention output + MLP output, do I recover the block output that Qwen actually produced?

And the answer is essentially **yes**.

You'd ideally get zero:

```text
rin + attention + MLP - rout ≈ 0
```

You got:

```text
0.0033
```

against an output magnitude of:

```text
10.437
```

so the discrepancy is tiny (~0.03%), consistent with numerical precision / implementation details.

That experimentally verifies the architecture rather than merely trusting a diagram.

### One nuance that matters

The simplified picture is:

$$r_{\text{out}} \approx r_{\text{in}} + \mathrm{Attn} + \mathrm{MLP}$$

but Qwen's internal computation isn't literally just feeding raw `rin` directly into both modules. Normalization happens around the sublayers. Conceptually it's closer to:

$$
a = \mathrm{Attention}(\mathrm{Norm}(r_{\text{in}}))
$$

$$
r' = r_{\text{in}} + a
$$

$$
m = \mathrm{MLP}(\mathrm{Norm}(r'))
$$

$$
r_{\text{out}} = r' + m
$$

Your hooks are capturing the **writes after those internal transformations**, which is why the additive equation works.

And this gets us to the MI idea:

Right now you know only the **size** of each write:

```text
Attention write = 7.387
MLP write       = 4.594
```

You **do not yet know what information those vectors represent**.

A large attention write doesn't mean "attention understood disease progression." And a 4.594 MLP write doesn't mean the MLP encoded "FOLFOX failed."

You've learned:

> **WHERE and HOW MUCH the model wrote.**

MI now asks:

> **WHAT did it write?**

And causal MI eventually asks:

> **Did that write actually cause the model's answer?**

That's the bridge from this notebook into `difference → projection → SAE → ablation/patching → circuits`.